# Rencana Eksekusi Proyek AI: PSU091

## Tujuan Fase Eksekusi

Membangun model Deep Learning untuk optimasi menu makanan dengan batasan anggaran Rp10.000 menggunakan arsitektur TensorFlow tingkat lanjut.

### Pembagian Tugas Teknis (AI Engineer)

1. Tugas Yahdillah (Arsitektur & Core Logic)
Fokus pada pembangunan struktur utama dan logika sistem agar sesuai dengan standar penilaian Advanced Dicoding.

- Arsitektur Model: Merancang model menggunakan TensorFlow Functional API untuk fleksibilitas input nutrisi dan harga.

- Custom Training: Mengimplementasikan tf.GradientTape untuk proses update weights secara manual tanpa model.fit().

- Model Saving: Mengatur penyimpanan model ke format .keras agar siap digunakan oleh sistem Backend.

- Integrasi API: Menyelesaikan skrip FastAPI di folder /api sebagai jembatan antara model AI dan aplikasi web.

2. Tugas Silviyana (Data Preprocessing & Optimization)
Fokus pada persiapan data mentah dan memastikan model bekerja dengan performa terbaik.

- Data Wrangling: Mengolah dataset nutrisi dari tim Data Science ke dalam bentuk tensor yang siap diproses model.

- Feature Engineering: Melakukan scaling dan normalization pada fitur harga dan kalori bahan makanan.

- Hyperparameter Tuning: Melakukan eksperimen pada jumlah hidden layers, learning rate, dan batch size untuk meminimalkan Mean Absolute Error (MAE).

- Monitoring & Evaluasi: Membuat visualisasi grafik pelatihan (Loss & Accuracy) untuk memastikan model tidak mengalami overfitting.

In [6]:
!pip install tensorflow pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 92.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 77.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [tensorflow]5 [tensorflow]-py]


In [7]:
import tensorflow as tf
import numpy as np
import pandas as pd

# Cek apakah sistem sudah siap
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [8]:
def build_model():
    # Mendefinisikan input: Anggaran dan Target Kalori
    inputs = tf.keras.Input(shape=(2,), name="input_layer")
    
    # Hidden Layers (Lapisan tersembunyi untuk belajar)
    x = tf.keras.layers.Dense(128, activation='relu')(inputs)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    
    # Output Layer: Menghasilkan rekomendasi
    outputs = tf.keras.layers.Dense(1, name="output_layer")(x) 
    
    # Menggabungkan menjadi satu model
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    return model

my_model = build_model()
my_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 2)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,753 (42.00 KB)

 Trainable params: 10,753 (42.00 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_object = tf.keras.losses.MeanSquaredError()

@tf.function
def train_step(data_input, labels):
    with tf.GradientTape() as tape:
        predictions = my_model(data_input, training=True)
        loss = loss_object(labels, predictions) 
    gradients = tape.gradient(loss, my_model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, my_model.trainable_variables))
    
    return loss

In [12]:
# Membuat data percobaan (2 input: Budget & Kalori)
sample_input = tf.constant([[10000.0, 2000.0]], dtype=tf.float32)
sample_label = tf.constant([[1.0]], dtype=tf.float32) # Target nilai gizi/skor

# Memanggil fungsi yang sudah kamu buat tadi
current_loss = train_step(sample_input, sample_label)

# Menampilkan hasil error (loss) pertama
print(f"Hasil Loss Pertama: {current_loss.numpy()}")

Hasil Loss Pertama: 3580.808349609375
